<a href="https://colab.research.google.com/github/vulpecriptografica/Symm4ML/blob/main/lie_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lie Groups — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Lie Groups exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/lie_companion.ipynb)

## Setup

In [1]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [5]:
import numpy as np
from numpy import einsum as ein
from typing import List
import scipy as sp

from symm4ml import groups, linalg, rep, lie

### Reference data

These structure constants and generators are used throughout the exercise for testing.

In [3]:
# SO(3) structure constants
so3_A = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) L=1 generators
so3_X = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(1,3) structure constants
so13_A = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
        ],
        [
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SO(1,3) generators
so13_X = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 1.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SU(2) generators
su2_X = np.array(
    [
        [
            [0.0 + 0.0j, 0.5 + 0.0j],
            [-0.5 + 0.0j, 0.0 + 0.0j],
        ],
        [
            [-0.0 - 0.5j, 0.0 + 0.0j],
            [0.0 + 0.0j, 0.0 + 0.5j],
        ],
        [
            [0.0 - 0.0j, 0.0 + 0.5j],
            [0.0 + 0.5j, 0.0 - 0.0j],
        ],
    ]
)

print(f"so3_A: {so3_A.shape}, so3_X: {so3_X.shape}")
print(f"so13_A: {so13_A.shape}, so13_X: {so13_X.shape}")
print(f"su2_X: {su2_X.shape}")

so3_A: (3, 3, 3), so3_X: (3, 3, 3)
so13_A: (6, 6, 6), so13_X: (6, 4, 4)
su2_X: (3, 2, 2)


---
## 1. `are_isomorphic(X1, X2)`

Check if two representations of a Lie algebra (expressed in terms of generators) are isomorphic, i.e., the same up to a similarity transform. You can use a function from a previous problem set.

In [4]:
def are_isomorphic(X1: np.ndarray, X2: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if two representations of a Lie group are isomorphic.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        are_isomorphic: bool - True if the representations are isomorphic.
    """
    # U e^X U^-1 = e^(UXU^-1)
    # goal: get U from change of basis, then apply to e^X to see if it works for generators

    # # get U
    # U = linalg.infer_change_of_basis(X1, X2)
    # U_inv = np.linalg.inv(U)

    # # compare transformed X1 gen and X2
    # conj = U @ X1 @ U_inv
    # exp_1_trans = scipy.linalg.expm(conj)
    # exp_2 = scipy.linalg.expm(X2)

    e_x1 = scipy.linalg.expm(X1)
    e_x2 = scipy.linalg.expm(X1)

    return rep.are_isomorphic(e_x1, e_x2)

    # return np.allclose(exp_1_trans, exp_2)


are_isomorphic(so3_X, so3_X)

True

In [25]:
# Small tests from lie.py
assert are_isomorphic(so3_X, so3_X)
print("are_isomorphic tests passed!")

are_isomorphic tests passed!


In [5]:
dir(lie)


"\n    einsum(subscripts, *operands, out=None, dtype=None, order='K',\n           casting='safe', optimize=False)\n\n    Evaluates the Einstein summation convention on the operands.\n\n    Using the Einstein summation convention, many common multi-dimensional,\n    linear algebraic array operations can be represented in a simple fashion.\n    In *implicit* mode `einsum` computes these values.\n\n    In *explicit* mode, `einsum` provides further flexibility to compute\n    other array operations that might not be considered classical Einstein\n    summation operations, by disabling, or forcing summation over specified\n    subscript labels.\n\n    See the notes and examples for clarification.\n\n    Parameters\n    ----------\n    subscripts : str\n        Specifies the subscripts for summation as comma separated list of\n        subscript labels. An implicit (classical Einstein summation)\n        calculation is performed unless the explicit indicator '->' is\n        included as well 

---
## 2. `tensor_product(X1, X2)`

Compute the tensor product of two representations of a Lie algebra, where both input and output are expressed in terms of generators.

In [24]:
def tensor_product(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    """Tensor product of two representations of a Lie group.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        X: np.array [lie_dim, d1*d2, d1*d2] - tensor product of the representations.
    """
    d1 = X1.shape[1]
    d2 = X2.shape[1]
    X1_prom = np.kron(X1, np.eye(d2))
    X2_prom = np.kron(np.eye(d1), X2)
    return X1_prom + X2_prom

tensor_product(so3_X, so3_X)

array([[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1.,  0., -1., -0.],
        [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -1.],
        [ 0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  1.,  0.,  1.,  0.]],

       [[ 0.,  0.,  1.,  0.,  0.,  0.,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.],
        [-1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.],
        [-1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.],
        [ 0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [-0.,  0., -1.,  0.,  0.,  0., -1.,  0.,  0.]],

      

In [23]:
# Small tests from lie.py
assert tensor_product(so3_X, so3_X).shape == (3, 9, 9)
print("tensor_product tests passed!")

tensor_product tests passed!


---
## 3. `is_a_representation(algebra, X)`

Check whether the given generators `X` satisfy the commutation relations encoded in `algebra`: $[X_i, X_j] = \sum_k A_{ijk} X_k$.

In [57]:
def is_a_representation(
    algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8
) -> bool:
    """Check if X satisfies the commutation relations of the Lie algebra:
        [X_i, X_j] = sum_k A_{ijk} X_k
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_a_representation: bool - True if X satisfies the commutation relations.
    """
    num_gen = X.shape[0]
    for i in range(num_gen):
      for j in range(num_gen):
        lhs = X[i]@ X[j] - X[j]@ X[i]
        A_ij = algebra[i][j]
        print(A_ij)
        rhs = np.einsum('a, abc-> bc', A_ij, X)
        if np.allclose(lhs, rhs):
          continue
        else:
          return False
    return True

is_a_representation(so3_A, so3_X)

[0. 0. 0.]
[0. 0. 1.]
[ 0. -1.  0.]
[ 0.  0. -1.]
[0. 0. 0.]
[1. 0. 0.]
[0. 1. 0.]
[-1.  0.  0.]
[0. 0. 0.]


True

In [39]:
# Small tests from lie.py
assert is_a_representation(so3_A, so3_X)
print("is_a_representation tests passed!")

[[[ 0.  0.  0.]
  [ 0.  0.  1.]
  [ 0. -1.  0.]]

 [[ 0.  0. -1.]
  [ 0.  0.  0.]
  [ 1.  0.  0.]]

 [[ 0.  1.  0.]
  [-1.  0.  0.]
  [ 0.  0.  0.]]]
lhs: [[[ 0. -1.  0.]
  [ 1.  0.  0.]
  [ 0.  0.  0.]]

 [[ 0.  0. -1.]
  [ 0.  0.  0.]
  [ 1.  0.  0.]]]


NameError: name 'X_k' is not defined

---
## 4. `is_an_irrep(algebra, X)`

Check if a representation of a Lie algebra is irreducible. Return `True` if the input is a valid representation AND is irreducible.

In [40]:
def is_an_irrep(algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if X is an irreducible representation of the Lie algebra.
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_an_irrep: bool - True if X is an irreducible representation.
    """
    # YOUR CODE HERE
    pass

In [41]:
# Small tests from lie.py
assert is_an_irrep(so3_A, so3_X)
print("is_an_irrep tests passed!")

AssertionError: 

---
## 5. `decompose_rep_into_irreps(X)`

Decompose a representation of a Lie algebra into a direct sum of irreducible representations. Follow the same algorithm as for finite groups.

In [42]:
def decompose_rep_into_irreps(X: np.array, *, tol: float = 1e-8) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        Ys: List[np.array] - list of generators of irreducible representations.
    """
    # YOUR CODE HERE
    pass

In [43]:
# Small tests from lie.py
assert {
    X.shape[1] for X in decompose_rep_into_irreps(tensor_product(so3_X, so3_X))
} == {1, 3, 5}
print("decompose_rep_into_irreps tests passed!")

TypeError: 'NoneType' object is not iterable

---
## 6. `infer_irreps_from_tensor_products(X, n)`

Infer `n` non-isomorphic finite-dimensional irreps of the underlying matrix Lie group by successively decomposing tensor products. Start with the trivial representation and take successive tensor products with `X` and (if needed) its conjugate `X*`.

In [44]:
def infer_irreps_from_tensor_products(
    X: np.ndarray, n: int, *, tol: float = 1e-8
) -> List[np.ndarray]:
    """Infers irreducible representations from successive tensor products of a representation.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
        n: int - number of non-isomorphic irreducible representations to infer.
    Output:
        Ys: List[np.array] - list of n generators of irreducible representations,
            each an np.array of shape [lie_dim, d', d'] for some d'.
    """
    # YOUR CODE HERE
    pass

In [45]:
# Small tests from lie.py
Xs = infer_irreps_from_tensor_products(so3_X, 3)
assert len(Xs) == 3
assert Xs[0].shape == (3, 1, 1)
assert is_an_irrep(so3_A, Xs[0])
assert Xs[1].shape == (3, 3, 3)
assert is_an_irrep(so3_A, Xs[1])
assert Xs[2].shape == (3, 5, 5)
assert is_an_irrep(so3_A, Xs[2])
print("infer_irreps_from_tensor_products tests passed!")

TypeError: object of type 'NoneType' has no len()

---
## Explore Further

In [46]:
# Try inferring irreps of SO(1,3) or SU(2)!